In [2]:
# SILVER LAYER

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    explode,
    from_unixtime,
    to_timestamp
)


#Create Spark session

spark = (
    SparkSession.builder
    .appName("Spotify Silver ETL")
    .config("spark.driver.memory", "24g")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-azure:3.4.2"
    )
    .config(
        "fs.azure.account.auth.type.spotifydestorage.dfs.core.windows.net",
        "OAuth"
    )
    .config(
        "fs.azure.account.oauth.provider.type.spotifydestorage.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.MsiTokenProvider"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


STORAGE_ACCOUNT = "spotifydestorage"
DATA_CONTAINER = "data"


bronze_path = (
    f"abfss://{DATA_CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/playlists.parquet"
)


silver_base_path = (
    f"abfss://{DATA_CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "silver/"
)

playlist_path = silver_base_path + "playlists.parquet"
artist_path = silver_base_path + "artist.parquet"
album_path = silver_base_path + "album.parquet"
tracks_path = silver_base_path + "tracks.parquet"
pos_bridge_path = silver_base_path + "pos_bridge.parquet"




# print("========================================")
# print("SPOTIFY SILVER ETL")
# print("========================================")

# print("\nBronze path:")
# print(bronze_path)

# print("\nSilver paths:")
# print(playlist_path)
# print(artist_path)
# print(album_path)
# print(tracks_path)
# print(pos_bridge_path)


#Read Bronze playlist 

print("\n========================================")
print("READING BRONZE PLAYLISTS")
print("========================================")

bronze_playlist = spark.read.parquet(bronze_path)

print("\nBronze playlist schema:")
bronze_playlist.printSchema()


:: loading settings :: url = jar:file:/home/linux_vm_user/spotifymp-project/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/linux_vm_user/.ivy2.5.2/cache
The jars for the packages stored in: /home/linux_vm_user/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7728fe9c-146d-4c3d-a9e3-b0d03807feed;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.2 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.3.0 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.4.0 in central
	found org.eclipse.jetty#


READING BRONZE PLAYLISTS

Bronze playlist schema:
root
 |-- collaborative: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- modified_at: long (nullable = true)
 |-- name: string (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- pid: long (nullable = true)
 |-- tracks: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album_name: string (nullable = true)
 |    |    |-- album_uri: string (nullable = true)
 |    |    |-- artist_name: string (nullable = true)
 |    |    |-- artist_uri: string (nullable = true)
 |    |    |-- duration_ms: long (nullable = true)
 |    |    |-- pos: long (nullable = true)
 |    |    |-- track_name: string (nullable = true)
 |    |    |-- track_uri: string (nullable = true)



In [3]:
# ============================================================
# 8. SILVER PLAYLIST TABLE
# ============================================================

print("\n========================================")
print("CREATING SILVER PLAYLIST TABLE")
print("========================================")

silver_playlist = (
    bronze_playlist
    .select(
        col("pid")
            .cast("long")
            .alias("pid"),

        col("name")
            .cast("string")
            .alias("name"),

        col("collaborative")
            .cast("boolean")
            .alias("collaborative"),

        to_timestamp(
            from_unixtime(
                col("modified_at").cast("long")
            )
        ).alias("modified_at"),

        col("num_tracks")
            .cast("long")
            .alias("num_tracks"),

        col("num_albums")
            .cast("long")
            .alias("num_albums"),

        col("num_followers")
            .cast("long")
            .alias("num_followers"),

        col("num_edits")
            .cast("long")
            .alias("num_edits"),

        col("num_artists")
            .cast("long")
            .alias("num_artists"),

        col("duration_ms")
            .cast("long")
            .alias("duration_ms"),

        col("description")
            .cast("string")
            .alias("description")
    )
)

print("\nSilver Playlist schema:")
silver_playlist.printSchema()


# ============================================================
# 9. Write Silver Playlist
# ============================================================

print("\nWriting Silver Playlist...")

(
    silver_playlist
    .write
    .mode("overwrite")
    .parquet(playlist_path)
)

print("Silver Playlist written successfully.")


# ============================================================
# 10. Explode tracks
# ============================================================
#
# Bronze playlists contain a nested "tracks" array.
#
# We explode the array once and use the resulting DataFrame
# to create:
#
#   Artist
#   Album
#   Tracks
#   POS Bridge
#
# ============================================================

print("\n========================================")
print("EXPLODING TRACKS")
print("========================================")

playlist_tracks = (
    bronze_playlist
    .select(
        col("pid")
            .cast("long")
            .alias("pid"),

        explode("tracks")
            .alias("track")
    )
)

print("\nExploded track schema:")
playlist_tracks.printSchema()



# ============================================================
# 19. VERIFY SILVER TABLES
# ============================================================

print("\n========================================")
print("VERIFYING SILVER TABLES")
print("========================================")


# ------------------------------------------------------------
# Playlist
# ------------------------------------------------------------

print("\n--- PLAYLIST ---")

check_playlist = spark.read.parquet(playlist_path)

check_playlist.printSchema()

print(f"Rows: {check_playlist.count()}")

check_playlist.show(
    5,
    truncate=False
)



CREATING SILVER PLAYLIST TABLE

Silver Playlist schema:
root
 |-- pid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- collaborative: boolean (nullable = true)
 |-- modified_at: timestamp (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- description: string (nullable = true)


Writing Silver Playlist...


Silver Playlist written successfully.

EXPLODING TRACKS

Exploded track schema:
root
 |-- pid: long (nullable = true)
 |-- track: struct (nullable = true)
 |    |-- album_name: string (nullable = true)
 |    |-- album_uri: string (nullable = true)
 |    |-- artist_name: string (nullable = true)
 |    |-- artist_uri: string (nullable = true)
 |    |-- duration_ms: long (nullable = true)
 |    |-- pos: long (nullable = true)
 |    |-- track_name: string (nullable = true)
 |    |-- track_uri: string (nullable = true)


VERIFYING SILVER TABLES

--- PLAYLIST ---
root
 |-- pid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- collaborative: boolean (nullable = true)
 |-- modified_at: timestamp (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- description: string (

Rows: 1000000
+------+--------------+-------------+-------------------+----------+----------+-------------+---------+-----------+-----------+-----------+
|pid   |name          |collaborative|modified_at        |num_tracks|num_albums|num_followers|num_edits|num_artists|duration_ms|description|
+------+--------------+-------------+-------------------+----------+----------+-------------+---------+-----------+-----------+-----------+
|742000|Party hard    |false        |2017-10-24 00:00:00|43        |40        |1            |3        |37         |9621217    |NULL       |
|742001|Reggae        |false        |2017-03-29 00:00:00|37        |30        |1            |6        |18         |8230661    |NULL       |
|742002|Everything    |false        |2017-10-15 00:00:00|10        |5         |1            |2        |3          |2025344    |NULL       |
|742003|one           |false        |2017-05-30 00:00:00|160       |120       |2            |9        |84         |34797052   |NULL       |
|74200

In [4]:
# ============================================================
# 11. SILVER ARTIST TABLE
# ============================================================

print("\n========================================")
print("CREATING SILVER ARTIST TABLE")
print("========================================")

silver_artist = (
    playlist_tracks
    .select(
        col("track.artist_uri")
            .cast("string")
            .alias("artist_uri"),

        col("track.artist_name")
            .cast("string")
            .alias("artist_name")
    )
    .dropDuplicates(["artist_uri"])
)

print("\nSilver Artist schema:")
silver_artist.printSchema()


# ============================================================
# 12. Write Silver Artist
# ============================================================

print("\nWriting Silver Artist...")

(
    silver_artist
    .write
    .mode("overwrite")
    .parquet(artist_path)
)

print("Silver Artist written successfully.")



# ------------------------------------------------------------
# Artist
# ------------------------------------------------------------

print("\n--- ARTIST ---")

check_artist = spark.read.parquet(artist_path)

check_artist.printSchema()

print(f"Rows: {check_artist.count()}")

check_artist.show(
    5,
    truncate=False
)





CREATING SILVER ARTIST TABLE

Silver Artist schema:
root
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)


Writing Silver Artist...


Silver Artist written successfully.

--- ARTIST ---
root
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)

Rows: 295860
+-------------------------------------+--------------------------+
|artist_uri                           |artist_name               |
+-------------------------------------+--------------------------+
|spotify:artist:0001cekkfdEBoMlwVQvpLg|Jordan Colle              |
|spotify:artist:0004C5XZIKZyd2RWvP4sOq|"Faron Young, Nat Stuckey"|
|spotify:artist:000Dq0VqTZpxOP6jQMscVL|Thug Brothers             |
|spotify:artist:000UUAlAdQqkTD9sfoyQGf|Darren Gibson             |
|spotify:artist:001NaPzWk3GKPMkuVNMdGd|Cuzzo Shay                |
+-------------------------------------+--------------------------+
only showing top 5 rows


In [5]:

# ============================================================
# 13. SILVER ALBUM TABLE
# ============================================================

print("\n========================================")
print("CREATING SILVER ALBUM TABLE")
print("========================================")

silver_album = (
    playlist_tracks
    .select(
        col("track.album_uri")
            .cast("string")
            .alias("album_uri"),

        col("track.album_name")
            .cast("string")
            .alias("album_name")
    )
    .dropDuplicates(["album_uri"])
)

print("\nSilver Album schema:")
silver_album.printSchema()


# ============================================================
# 14. Write Silver Album
# ============================================================

print("\nWriting Silver Album...")

(
    silver_album
    .write
    .mode("overwrite")
    .parquet(album_path)
)

print("Silver Album written successfully.")



# ------------------------------------------------------------
# Album
# ------------------------------------------------------------

print("\n--- ALBUM ---")

check_album = spark.read.parquet(album_path)

check_album.printSchema()

print(f"Rows: {check_album.count()}")

check_album.show(
    5,
    truncate=False
)




CREATING SILVER ALBUM TABLE

Silver Album schema:
root
 |-- album_uri: string (nullable = true)
 |-- album_name: string (nullable = true)


Writing Silver Album...


Silver Album written successfully.

--- ALBUM ---
root
 |-- album_uri: string (nullable = true)
 |-- album_name: string (nullable = true)

Rows: 734684
+------------------------------------+------------------------------------+
|album_uri                           |album_name                          |
+------------------------------------+------------------------------------+
|spotify:album:00045VFusrXwCSietfmspc|Let Love Begin Remixed              |
|spotify:album:0005lpYtyKk9B3e0mWjdem|Stability                           |
|spotify:album:0005rH90S3le891y5XzPg4|Mozart: Piano Concerto No. 27, KV595|
|spotify:album:0008WZMLnvEBVnq418uZsI|Smart Flesh                         |
|spotify:album:000VWSwMyH76POuqUWqF48|Ibiza ChillOut Classics             |
+------------------------------------+------------------------------------+
only showing top 5 rows


In [6]:

# ============================================================
# 15. SILVER TRACKS TABLE
# ============================================================

print("\n========================================")
print("CREATING SILVER TRACKS TABLE")
print("========================================")

silver_tracks = (
    playlist_tracks
    .select(
        col("track.track_uri")
            .cast("string")
            .alias("track_uri"),

        col("track.track_name")
            .cast("string")
            .alias("track_name"),

        col("track.artist_uri")
            .cast("string")
            .alias("artist_uri"),

        col("track.duration_ms")
            .cast("long")
            .alias("duration_ms")
    )
    .dropDuplicates(["track_uri"])
)

print("\nSilver Tracks schema:")
silver_tracks.printSchema()


# ============================================================
# 16. Write Silver Tracks
# ============================================================

print("\nWriting Silver Tracks...")

(
    silver_tracks
    .write
    .mode("overwrite")
    .parquet(tracks_path)
)

print("Silver Tracks written successfully.")



# ------------------------------------------------------------
# Tracks
# ------------------------------------------------------------

print("\n--- TRACKS ---")

check_tracks = spark.read.parquet(tracks_path)

check_tracks.printSchema()

print(f"Rows: {check_tracks.count()}")

check_tracks.show(
    5,
    truncate=False
)








CREATING SILVER TRACKS TABLE

Silver Tracks schema:
root
 |-- track_uri: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- duration_ms: long (nullable = true)


Writing Silver Tracks...


Silver Tracks written successfully.

--- TRACKS ---
root
 |-- track_uri: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- duration_ms: long (nullable = true)

Rows: 2262292
+------------------------------------+-------------------------------------------+-------------------------------------+-----------+
|track_uri                           |track_name                                 |artist_uri                           |duration_ms|
+------------------------------------+-------------------------------------------+-------------------------------------+-----------+
|spotify:track:000HmzbYBg0Uxe6cE47Tws|Exhuming McCarthy - Live                   |spotify:artist:4KWTAlx2RvbpseOGMEmROg|193547     |
|spotify:track:000IBgSmsLoeNcOuyXnoIE|Way Home - Slow Hands & Cameo Culture Remix|spotify:artist:5LrEhvvLmUxuzqCQdJlwnV|455173     |
|spotify:track:000U6DuGqCOrzL218D1yVi|Black & Yellow                             |spotify:artist:

In [7]:

# ============================================================
# 17. SILVER POS BRIDGE TABLE
# ============================================================

print("\n========================================")
print("CREATING SILVER POS BRIDGE TABLE")
print("========================================")

silver_pos_bridge = (
    playlist_tracks
    .select(
        col("pid")
            .cast("long")
            .alias("pid"),

        col("track.pos")
            .cast("long")
            .alias("pos"),

        col("track.track_uri")
            .cast("string")
            .alias("track_uri")
    )
)

print("\nSilver POS Bridge schema:")
silver_pos_bridge.printSchema()


# ============================================================
# 18. Write Silver POS Bridge
# ============================================================

print("\nWriting Silver POS Bridge...")

(
    silver_pos_bridge
    .write
    .mode("overwrite")
    .parquet(pos_bridge_path)
)

print("Silver POS Bridge written successfully.")




# ------------------------------------------------------------
# POS Bridge
# ------------------------------------------------------------

print("\n--- POS BRIDGE ---")

check_pos_bridge = spark.read.parquet(pos_bridge_path)

check_pos_bridge.printSchema()

print(f"Rows: {check_pos_bridge.count()}")

check_pos_bridge.show(
    5,
    truncate=False
)





CREATING SILVER POS BRIDGE TABLE

Silver POS Bridge schema:
root
 |-- pid: long (nullable = true)
 |-- pos: long (nullable = true)
 |-- track_uri: string (nullable = true)


Writing Silver POS Bridge...


Silver POS Bridge written successfully.

--- POS BRIDGE ---
root
 |-- pid: long (nullable = true)
 |-- pos: long (nullable = true)
 |-- track_uri: string (nullable = true)



Rows: 66346428
+------+---+------------------------------------+
|pid   |pos|track_uri                           |
+------+---+------------------------------------+
|183000|0  |spotify:track:7eaKWfov7b2Qa2n6HTesL3|
|183000|1  |spotify:track:57hCSu4zMTdnSum7NBL1Ye|
|183000|2  |spotify:track:7lL3MvFWFFSD25pBz72Agj|
|183000|3  |spotify:track:1uDjaezEbalGyGnuH80zDK|
|183000|4  |spotify:track:79XrkTOfV1AqySNjVlygpW|
+------+---+------------------------------------+
only showing top 5 rows


In [8]:

# ============================================================
# 20. PRIMARY KEY UNIQUENESS CHECKS
# ============================================================

print("\n========================================")
print("CHECKING PRIMARY KEY UNIQUENESS")
print("========================================")


# ------------------------------------------------------------
# Playlist - pid
# ------------------------------------------------------------

playlist_total = check_playlist.count()

playlist_distinct = (
    check_playlist
    .select("pid")
    .distinct()
    .count()
)

print("\nPlaylist pid:")
print(f"Total rows:     {playlist_total}")
print(f"Distinct pid:   {playlist_distinct}")

if playlist_total == playlist_distinct:
    print("PASSED")
else:
    print("WARNING: Duplicate pid values found.")


# ------------------------------------------------------------
# Artist - artist_uri
# ------------------------------------------------------------

artist_total = check_artist.count()

artist_distinct = (
    check_artist
    .select("artist_uri")
    .distinct()
    .count()
)

print("\nArtist artist_uri:")
print(f"Total rows:     {artist_total}")
print(f"Distinct URI:   {artist_distinct}")

if artist_total == artist_distinct:
    print("PASSED")
else:
    print("WARNING: Duplicate artist_uri values found.")


# ------------------------------------------------------------
# Album - album_uri
# ------------------------------------------------------------

album_total = check_album.count()

album_distinct = (
    check_album
    .select("album_uri")
    .distinct()
    .count()
)

print("\nAlbum album_uri:")
print(f"Total rows:     {album_total}")
print(f"Distinct URI:   {album_distinct}")

if album_total == album_distinct:
    print("PASSED")
else:
    print("WARNING: Duplicate album_uri values found.")


# ------------------------------------------------------------
# Tracks - track_uri
# ------------------------------------------------------------

tracks_total = check_tracks.count()

tracks_distinct = (
    check_tracks
    .select("track_uri")
    .distinct()
    .count()
)

print("\nTracks track_uri:")
print(f"Total rows:     {tracks_total}")
print(f"Distinct URI:   {tracks_distinct}")

if tracks_total == tracks_distinct:
    print("PASSED")
else:
    print("WARNING: Duplicate track_uri values found.")


# ------------------------------------------------------------
# POS Bridge - (pid, pos)
# ------------------------------------------------------------

pos_bridge_total = check_pos_bridge.count()

pos_bridge_distinct = (
    check_pos_bridge
    .select("pid", "pos")
    .distinct()
    .count()
)

print("\nPOS Bridge (pid, pos):")
print(f"Total rows:       {pos_bridge_total}")
print(f"Distinct pairs:   {pos_bridge_distinct}")

if pos_bridge_total == pos_bridge_distinct:
    print("PASSED")
else:
    print("WARNING: Duplicate (pid, pos) values found.")



CHECKING PRIMARY KEY UNIQUENESS



Playlist pid:
Total rows:     1000000
Distinct pid:   1000000
PASSED

Artist artist_uri:
Total rows:     295860
Distinct URI:   295860
PASSED

Album album_uri:
Total rows:     734684
Distinct URI:   734684
PASSED



Tracks track_uri:
Total rows:     2262292
Distinct URI:   2262292
PASSED



POS Bridge (pid, pos):
Total rows:       66346428
Distinct pairs:   66346428
PASSED


In [ ]:

# ============================================================
# 21. FINAL MESSAGE
# ============================================================

print("\n========================================")
print("ALL SILVER TABLES CREATED SUCCESSFULLY")
print("========================================")

print("\nSilver tables:")

print(f"Playlist:    {playlist_path}")
print(f"Artist:      {artist_path}")
print(f"Album:       {album_path}")
print(f"Tracks:      {tracks_path}")
print(f"POS Bridge:  {pos_bridge_path}")


# ============================================================
# 22. Stop Spark
# ============================================================

spark.stop()

print("\n========================================")
print("SILVER ETL COMPLETE")
print("========================================")


ALL SILVER TABLES CREATED SUCCESSFULLY

Silver tables:
Playlist:    abfss://data@spotifydestorage.dfs.core.windows.net/silver/playlists.parquet
Artist:      abfss://data@spotifydestorage.dfs.core.windows.net/silver/artist.parquet
Album:       abfss://data@spotifydestorage.dfs.core.windows.net/silver/album.parquet
Tracks:      abfss://data@spotifydestorage.dfs.core.windows.net/silver/tracks.parquet
POS Bridge:  abfss://data@spotifydestorage.dfs.core.windows.net/silver/pos_bridge.parquet

SILVER ETL COMPLETE
